# COMP5339 Assignment 2

GROUP: TUT11-ASSIGNMENTGRP-10 <br>

SID <br>
- 540969766
- 540931475

In [18]:
import pandas as pd
import numpy as np
import requests
import time
from datetime import datetime, timedelta
from dotenv import load_dotenv
import os
import ast


## Data Retrieval

In [2]:
# Load environment variables
load_dotenv()
API_KEY = os.getenv("API_KEY")

# Error handling for API key
if not API_KEY:
    raise ValueError("API_KEY is not set in the environment variable. Check .env file.")

base_url = "https://api.openelectricity.org.au/v4"

# Get the data from the API
def get_facilities():
    url = f'{base_url}/facilities/'
    params = {
        'interval': '5m',
        'network_id': 'NEM'
    }

    # Authorisation
    headers = {'Authorization': f'Bearer {API_KEY}'}

    # Make the request
    response = requests.get(url, headers = headers, params = params)
    facilities = pd.DataFrame(response.json())
    facilities.to_csv('facilities.csv', index = False)
    time.sleep(60) # wait for 60 seconds before the next request

    return facilities
    

In [3]:
# Get the metadata from the API
# get_facilities()
facilities = pd.read_csv('facilities.csv')
facilities.head()


,version,created_at,success,data,total_records
0,4.3.0,2025-10-27T15:08:28+11:00,True,"{'code': 'ADP', 'name': 'Adelaide Desalination...",514
1,4.3.0,2025-10-27T15:08:28+11:00,True,"{'code': 'ALDGASF', 'name': 'Aldoga', 'network...",514
2,4.3.0,2025-10-27T15:08:28+11:00,True,"{'code': 'AMCORGR', 'name': 'Amcor Glass', 'ne...",514
3,4.3.0,2025-10-27T15:08:28+11:00,True,"{'code': 'ANGASTON', 'name': 'Angaston', 'netw...",514
4,4.3.0,2025-10-27T15:08:28+11:00,True,"{'code': 'APS', 'name': 'Anglesea', 'network_i...",514


### Preprocessing for metadata

get_facilities 
일단 먼저 facilities 데이터를 가져와서
- facility_code 를 기준으로 name, location을 추출
  - 그리고 그 아래 unit_id 를 기준으로 code, fueltech_id, status_id, capacity_registered, capacity_maximum, dispatch_type를 정리
- unique facility_code를 list로 저장

get_facility_power
- 위에서 저장한 facility_code list 기준으로 원하는 기간의 power/energy 데이터 가져오기
- 그래서 새로운 table에는 facility_code, facility_name, location

In [38]:
class Dataloader:
    '''
    Handle data loading and preprocessing
    '''
    def __init__(self):
        '''
        Initialize the Dataloader with the API key  
        '''
        load_dotenv()
        api_key = os.getenv("API_KEY")

        if not api_key:
            raise ValueError("API_KEY is not set in the environment variable. Check .env file.")
        
        self.api_key = api_key
        self.base_url = "https://api.openelectricity.org.au/v4"
        self.headers = {'Authorization': f'Bearer {api_key}'}

        print("Ready to load data")
    
    def save_to_csv(self, df, filename):
        '''
        Save the dataframe to a csv file
        '''
        if not df.empty:
            df.to_csv(filename, index = False)
            print(f'Successfully saved data to {filename}')
        else:
            print("The dataframe is empty. No data to save.")


    def get_facilities(self):
        ''' 
        Get the facilities data from the API
        '''
        facilities_url = f'{self.base_url}/facilities/'
        params = {
            'network_id': 'NEM'
        }

        try:
            response = requests.get(facilities_url, headers = self.headers, params = params)
            facilities = pd.DataFrame(response.json())

            print(f'Successfully retrieved data for {len(facilities)} facilities')

            self.save_to_csv(facilities, 'facilities_raw.csv')
            
            time.sleep(60) # wait for 60 seconds before the next request

            return facilities
        
        except requests.exceptions.RequestException as e:
            print(f"Error occurred: {e}")
            return None
        

    def cleaning_facilities(self, facilities):
        '''
        Clean the facilities data
        '''
        cleaned_facilities = []

        for idx, row in facilities.iterrows():

            try:
                facility_data = ast.literal_eval(row['data']) # string to dict

                # Extract the basic information first
                facility_code = facility_data.get('code')
                facility_name = facility_data.get('name')
                facility_region = facility_data.get('network_region')

                # Extract the location
                location = facility_data.get('location', {})
                latitude = location.get('lat')
                longitude = location.get('lng')

                # Get the unit details
                units = facility_data.get('units', [])


                for unit in units:
                    unit_info = {
                        # Facility Unit information
                        'facility_code': facility_code,
                        'facility_name': facility_name,
                        'facility_region': facility_region,
                        'latitude': latitude,
                        'longitude': longitude,

                        # Facility power, emissions
                        'power_output': 0,
                        'emissions': 0,
                        
                        # Unit information
                        'unit_code': unit.get('code'),
                        'unit_status': unit.get('status_id'),
                        'fuel_type': unit.get('fueltech_id'),
                        'capacity_registered': unit.get('capacity_registered'),
                        'capacity_maximum': unit.get('capacity_maximum'),
                        'emissions_co2': unit.get('emissions_factor_co2'),

                        # Unit power, emissions
                        'unit_power': 0,
                        'unit_emissions': 0                      
                    }

                    cleaned_facilities.append(unit_info)
            
            except Exception as e:
                print(f"Error processing facility {idx}: {e}")
                continue


        return pd.DataFrame(cleaned_facilities)
            


        
        

    def get_facility_details(self, facility_code, interval, start_date, end_date):
        '''
        Get the facility power generation data
        '''
        facility_url = f'{self.base_url}/data/facilities/NEM'
        params = {
            'metrics': ['energy', 'emissions'],
            'interval': interval,
            'date_start': start_date,
            'date_end': end_date,
            'facility_code': facility_code
        }

        try:
            response = requests.get(facility_url, headers = self.headers, params = params)

            time.sleep(60)

            facility_data = response.json()


            records = {}

            if 'data' in facility_data:
                for metrics in facility_data['data']:
                    metric_name = metrics.get('metric') # Get energy or emissions

                    for record in metrics.get('results', []):
                        unit_code = record['columns']['unit_code']

                        for timestamp, value in record['data']:
                            key = (timestamp, unit_code)

                            if key not in records:
                                records[key] = {
                                    'interval': timestamp,
                                    'unit_code': unit_code,
                                    'unit_power': 0,
                                    'unit_emissions': 0
                                }
                            
                            if metric_name == 'energy':
                                records[key]['unit_power'] = value
                            elif metric_name == 'emissions':
                                records[key]['unit_emissions'] = value

            
            unit_data = pd.DataFrame(list(records.values()))

            if not unit_data.empty:
                print(f'Successfully retrieved {len(unit_data)} records for facility {unit_data['unit_code'].nunique()} units from {len(facility_code)} facilities!')

            return unit_data
        
        except requests.exceptions.RequestException as e:
            print(f"Error occurred: {e}")
            return None
        
            
            



In [4]:
loader = Dataloader()

facilities = loader.get_facilities()
facilities.head()

Ready to load data
Successfully retrieved data for 514 facilities
Successfully saved data to facilities_raw.csv


,version,created_at,success,data,total_records
0,4.3.0,2025-10-28T15:42:27+11:00,True,"{'code': 'ADP', 'name': 'Adelaide Desalination...",514
1,4.3.0,2025-10-28T15:42:27+11:00,True,"{'code': 'ALDGASF', 'name': 'Aldoga', 'network...",514
2,4.3.0,2025-10-28T15:42:27+11:00,True,"{'code': 'AMCORGR', 'name': 'Amcor Glass', 'ne...",514
3,4.3.0,2025-10-28T15:42:27+11:00,True,"{'code': 'ANGASTON', 'name': 'Angaston', 'netw...",514
4,4.3.0,2025-10-28T15:42:27+11:00,True,"{'code': 'APS', 'name': 'Anglesea', 'network_i...",514


In [39]:
facilities = pd.read_csv('facilities_raw.csv')

loader = Dataloader()

samples = facilities[0:10]

fac = loader.cleaning_facilities(samples)
facility_codes = fac['facility_code'].unique()

sample_units = loader.get_facility_details(facility_codes, '1d', '2025-10-01T00:00:00', '2025-10-07T23:59:59')



Ready to load data
Successfully retrieved 60 records for facility 10 units from 10 facilities!


In [37]:
fac

,facility_code,facility_name,facility_region,latitude,longitude,power_output,emissions,unit_code,unit_status,fuel_type,capacity_registered,capacity_maximum,emissions_co2,unit_power,unit_emissions
0,ADP,Adelaide Desalination,SA1,-35.096948,138.484061,0,0,ADPPV1,operating,solar_utility,24.75,19.00,NaN,0,0
1,ADP,Adelaide Desalination,SA1,-35.096948,138.484061,0,0,ADPPV2,operating,solar_utility,0.20,0.20,NaN,0,0
2,ADP,Adelaide Desalination,SA1,-35.096948,138.484061,0,0,ADPPV3,operating,solar_utility,0.02,0.02,NaN,0,0
3,ADP,Adelaide Desalination,SA1,-35.096948,138.484061,0,0,ADPBA1G,operating,battery_discharging,7.76,6.15,NaN,0,0
4,ADP,Adelaide Desalination,SA1,-35.096948,138.484061,0,0,ADPBA1L,operating,battery_charging,7.76,6.15,NaN,0,0
5,ADP,Adelaide Desalination,SA1,-35.096948,138.484061,0,0,ADPBA1,operating,battery,7.76,6.15,NaN,0,0
6,ALDGASF,Aldoga,QLD1,-23.839544,151.084900,0,0,ALDGASF1,operating,solar_utility,535.21,387.00,NaN,0,0
7,AMCORGR,Amcor Glass,SA1,-34.882663,138.577975,0,0,AMCORGR,retired,distillate,4.00,NaN,0.9000,0,0
8,ANGASTON,Angaston,SA1,-34.503948,139.024296,0,0,ANGAS1,retired,distillate,30.00,NaN,1.0136,0,0
9,ANGASTON,Angaston,SA1,-34.503948,139.024296,0,0,ANGAS2,retired,distillate,20.00,NaN,1.0136,0,0


In [40]:
sample_units

,interval,unit_code,unit_power,unit_emissions
0,2025-10-01T00:00:00+10:00,ADPBA1,2.4519,0.0000
1,2025-10-02T00:00:00+10:00,ADPBA1,-11.6253,0.0000
2,2025-10-03T00:00:00+10:00,ADPBA1,2.3180,0.0000
3,2025-10-04T00:00:00+10:00,ADPBA1,-1.4793,0.0000
4,2025-10-05T00:00:00+10:00,ADPBA1,-8.4774,0.0000
5,2025-10-06T00:00:00+10:00,ADPBA1,-2.4875,0.0000
6,2025-10-01T00:00:00+10:00,ADPBA1G,8.1476,0.0000
7,2025-10-02T00:00:00+10:00,ADPBA1G,9.2264,0.0000
8,2025-10-03T00:00:00+10:00,ADPBA1G,12.7921,0.0000
9,2025-10-04T00:00:00+10:00,ADPBA1G,15.7851,0.0000


## Data Integration and Materialisation/Cashing

- whether the per-facility power generated is power or energy
  - power: Instantaneous power output/consumption (MW)
  - energy: Energy generated/consumed over time (MWh)

- per-market price and demand -> is this about price and demand per region (NSW, SA) or per fuel type

- for visualisation:
  - is it okay to show powerstation name and current power output or emissions or should we have to keep those information floating
  - the latest power production and emissions data meaning the overall data for that certain 